In [9]:
!pip install torch gymnasium

In [10]:
# Import standard Python math libraries
import random
# Import numpy for numerical manipulations
import numpy as np
# Import PyTorch for deep learning
import torch
import torch.nn as nn
import torch.optim as optim
# Import Gymnasium to run the CartPole simulation
import gymnasium as gym

In [11]:
# Step 1: Create a basic neural network to estimate Q-values
class DQNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DQNetwork, self).__init__()
        # 2-layer simple MLP network
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        return self.fc(x)

In [12]:
# Step 2: Create a memory buffer that samples based on error weight priorities
class SimplePERBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity  # Maximum size of memory
        self.alpha = alpha        # Priority exponent factor
        self.buffer = []          # List to hold experiences
        self.priorities = []      # List to hold priority values

    def add(self, transition, error):
        # Calculate initial priority based on TD-error magnitude
        priority = (abs(error) + 1e-5) ** self.alpha
        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
            self.priorities.append(priority)
        else:
            # Overwrite oldest elements when full
            idx = len(self.buffer) % self.capacity
            self.buffer[idx] = transition
            self.priorities[idx] = priority

    def sample(self, batch_size):
        # Normalize priorities to construct a probability distribution
        probs = np.array(self.priorities) / sum(self.priorities)
        # Sample random indices based on calculated probability weight distributions
        indices = np.random.choice(len(self.buffer), batch_size, p=probs)
        samples = [self.buffer[idx] for idx in indices]
        return samples, indices

    def update_priorities(self, indices, errors):
        for idx, error in zip(indices, errors):
            # Update priorities based on new learning feedback
            self.priorities[idx] = (abs(error) + 1e-5) ** self.alpha

In [13]:
# Initialize CartPole environment
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

# Initialize network, target network, optimizer, and prioritized memory
policy_net = DQNetwork(state_dim, action_dim)
target_net = DQNetwork(state_dim, action_dim)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
memory = SimplePERBuffer(capacity=1000)

In [14]:
# Dummy test loop to demonstrate one single prioritize training update
raw_state, _ = env.reset()
action = env.action_space.sample()
next_raw_state, reward, terminated, truncated, _ = env.step(action)
done = terminated or truncated

# Convert state arrays to PyTorch tensors
state_t = torch.FloatTensor(raw_state)
next_state_t = torch.FloatTensor(next_raw_state)

# Calculate dynamic TD-error for initial priority estimation
with torch.no_grad():
    current_q = policy_net(state_t)[action].item()
    max_next_q = target_net(next_state_t).max().item()
    target_q = reward + 0.99 * max_next_q * (1 - int(done))
    initial_error = target_q - current_q

# Save transition to our prioritized buffer
memory.add((raw_state, action, reward, next_raw_state, done), initial_error)

In [15]:
# Sample a mini-batch of size 1 (since we only have 1 item in memory for demonstration)
samples, indices = memory.sample(batch_size=1)
s, a, r, ns, d = samples[0]

# Perform a basic training gradient step
pred_q = policy_net(torch.FloatTensor(s))[a]
with torch.no_grad():
    target_q = r + 0.99 * target_net(torch.FloatTensor(ns)).max() * (1 - int(d))

loss = nn.MSELoss()(pred_q, target_q)
optimizer.zero_grad()
loss.backward()
optimizer.step()

# Update buffer entry with the newly computed absolute error
new_error = abs((target_q - pred_q).item())
memory.update_priorities(indices, [new_error])

print("DQN with Prioritized Experience Replay initialized and updated successfully!")

DQN with Prioritized Experience Replay initialized and updated successfully!
